# Model Training — Churn Prediction (Telco Customer Churn)

Tercer notebook del pipeline (después de `01_eda.ipynb` y `02_feature_engineering.ipynb`).
Objetivo: entrenar y comparar Logistic Regression, Random Forest y XGBoost sobre el dataset limpio, elegir el mejor modelo por AUC, y subirlo a S3 como un único artefacto autocontenido (`models/churn_best_model.joblib`), listo para `src/deploy_lambda.py`.

Este notebook replica la lógica de `src/train.py` — mismo preprocesador, mismos modelos, mismas métricas — para poder iterar visualmente antes de correr el script en producción.

## 1. Setup e imports

In [ ]:
import sys
sys.path.append("../src")

import io
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, RocCurveDisplay, classification_report,
)
from xgboost import XGBClassifier

from s3_utils import read_csv_from_s3, get_s3_client, BUCKET_NAME

sns.set_theme(style="whitegrid")


## 2. Cargar el dataset procesado

Se lee directamente desde S3 el CSV limpio que dejó `02_feature_engineering.ipynb` (`processed/telco_churn_clean.csv`) — sin encoding todavía, eso se hace dentro del Pipeline.

In [ ]:
PROCESSED_KEY = "processed/telco_churn_clean.csv"
TARGET_COL = "Churn"

df = read_csv_from_s3(PROCESSED_KEY)
df.shape


## 3. Split y preprocesador

Mismo criterio que `src/train.py`: split 80/20 estratificado, `StandardScaler` para numéricas, `OneHotEncoder` para categóricas.

In [ ]:
y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

X_train.shape, X_test.shape


## 4. Definir candidatos

Tres modelos con complejidad creciente: uno lineal como baseline interpretable, y dos de ensamble que suelen capturar mejor las interacciones no lineales entre variables.

In [ ]:
candidates = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "xgboost": XGBClassifier(eval_metric="logloss", random_state=42),
}


## 5. Entrenar y evaluar cada modelo

Cada candidato se envuelve en un `Pipeline` junto con el preprocesador, para que el fit/transform quede encapsulado y no haya fuga de información. Se registran AUC, F1, precisión y recall.

In [ ]:
def evaluate(model, X_test, y_test) -> dict:
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    return {
        "auc": roc_auc_score(y_test, probs),
        "f1": f1_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
    }


results = {}
fitted_pipelines = {}

for name, clf in candidates.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", clf)])
    pipeline.fit(X_train, y_train)
    metrics = evaluate(pipeline, X_test, y_test)
    results[name] = metrics
    fitted_pipelines[name] = pipeline
    print(f"{name}: {metrics}")


## 6. Comparación de métricas

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("auc", ascending=False)
results_df


In [ ]:
results_df[["auc", "f1", "precision", "recall"]].plot(
    kind="bar", figsize=(8, 5), ylim=(0, 1)
)
plt.title("Comparación de modelos — Churn Prediction")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()


## 7. Elegir el mejor modelo (por AUC)

AUC es la métrica principal porque el dataset está desbalanceado (~73% No / ~27% Yes) y no depende del umbral de clasificación elegido, a diferencia de accuracy.

In [ ]:
best_name = results_df.index[0]
best_pipeline = fitted_pipelines[best_name]
best_auc = results_df.loc[best_name, "auc"]

print(f"Mejor modelo: {best_name} (AUC={best_auc:.4f})")


## 8. Diagnóstico del mejor modelo: matriz de confusión y curva ROC

In [ ]:
preds = best_pipeline.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
axes[0].set_title(f"Matriz de confusión — {best_name}")
axes[0].set_xlabel("Predicho")
axes[0].set_ylabel("Real")

RocCurveDisplay.from_estimator(best_pipeline, X_test, y_test, ax=axes[1])
axes[1].set_title(f"Curva ROC — {best_name}")

plt.tight_layout()
plt.show()


In [ ]:
print(classification_report(y_test, preds, target_names=["No Churn", "Churn"]))


## 9. Importancia de features (si aplica)

Solo para modelos de árboles (`random_forest`, `xgboost`). Si el mejor modelo es `logistic_regression`, se muestran los coeficientes en su lugar.

In [ ]:
classifier = best_pipeline.named_steps["classifier"]
feature_names = best_pipeline.named_steps["preprocessor"].get_feature_names_out()

if hasattr(classifier, "feature_importances_"):
    importances = pd.Series(classifier.feature_importances_, index=feature_names)
    top_features = importances.sort_values(ascending=False).head(15)
elif hasattr(classifier, "coef_"):
    importances = pd.Series(classifier.coef_[0], index=feature_names)
    top_features = importances.abs().sort_values(ascending=False).head(15)
    top_features = importances[top_features.index]
else:
    top_features = None

if top_features is not None:
    plt.figure(figsize=(8, 6))
    top_features.sort_values().plot(kind="barh")
    plt.title(f"Top 15 features — {best_name}")
    plt.tight_layout()
    plt.show()


## 10. Guardar el mejor modelo en S3

Se serializa el `Pipeline` completo (preprocesador + clasificador) como un único artefacto, sin pasar por disco, listo para que `src/deploy_lambda.py` lo cargue directamente.

In [ ]:
MODEL_KEY = "models/churn_best_model.joblib"

buffer = io.BytesIO()
joblib.dump(best_pipeline, buffer)
buffer.seek(0)

s3 = get_s3_client()
s3.put_object(Bucket=BUCKET_NAME, Key=MODEL_KEY, Body=buffer.getvalue())
print(f"Modelo guardado en s3://{BUCKET_NAME}/{MODEL_KEY}")


## 11. Notas y próximos pasos

- El modelo ganador (por AUC) queda guardado como un único artefacto en `s3://<bucket>/models/churn_best_model.joblib`.
- Este mismo resultado se puede reproducir de forma no interactiva corriendo `src/train.py`, que implementa la misma lógica (`train_and_compare()` + `save_model_to_s3()`) sin necesidad de abrir un notebook — útil para automatizar el reentrenamiento.
- Siguiente paso: desplegar el modelo con `src/deploy_lambda.py` (función Lambda) o como SageMaker Endpoint, y documentar en el `README.md` las métricas finales y los costos estimados de la arquitectura elegida.
- Pendiente de decidir: si el dataset de churn justifica ajustar el umbral de clasificación (por defecto 0.5) para priorizar recall sobre precisión, dado que en un caso de negocio real suele ser más costoso no detectar un cliente que se va a ir que contactar a uno que no lo iba a hacer.